# MedGemma Reasoning Traces — How the Agent Thinks

**Model transparency and explainability in clinical AI decision support.**

> This notebook is generated by `_generate_reasoning_traces_notebook.py`. Do not edit manually.

### Why Reasoning Traces Matter

Clinical AI systems must be transparent. When a model recommends a treatment change or flags
a contradiction, clinicians need to understand *why*. Black-box recommendations erode trust
and make it impossible to verify correctness.

MedGemma uses internal **"thinking tokens"** — a structured reasoning process that occurs
before the model produces its visible output. This notebook exposes that reasoning process
during contradiction resolution, showing how the agent arrives at its conclusions.

### What This Notebook Covers

- **Strategies A-D:** GPU-based reasoning with MedGemma thinking tokens (CR-1 through CR-6)
- **Strategy E:** Deterministic Python logic with no thinking tokens (CR-7 through CR-11)
- **Confidence mapping:** How reasoning quality maps to alert confidence levels
- **Transparency design:** When to use AI reasoning vs deterministic rules

### Quick Start
1. Set runtime to **GPU -> A100** (Runtime -> Change runtime type)
2. Add **HF_TOKEN** to Colab Secrets (key icon in left sidebar)
3. *(Optional)* Add **GITHUB_TOKEN** for private repo install
4. **Run All** (Runtime -> Run all)


> **Disclaimer:** This notebook demonstrates a research prototype. All clinical outputs
> (severity scores, antibiotic recommendations, contradiction alerts, CXR analysis)
> are **AI-generated by MedGemma 1.5 4B** and have been validated only on synthetic data.
> This system is **not a medical device**, is not approved for clinical use, and must not
> be used for patient care decisions. See [DISCLAIMER.md](../DISCLAIMER.md) for full terms.


In [ ]:
# Cell 1: Install package + dependencies
import os

# Detect Colab vs local
try:
    from google.colab import userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Install from GitHub (requires GITHUB_TOKEN in Colab Secrets)
    try:
        github_token = userdata.get("GITHUB_TOKEN")
        repo_url = f"git+https://{github_token}@github.com/HP-00/CAP-CDSS-MedGemma.git@main"
    except Exception:
        repo_url = "git+https://github.com/HP-00/CAP-CDSS-MedGemma.git@main"

    %pip install --quiet {repo_url}
    %pip install --quiet plotly>=5.0.0 pandas>=2.0.0 nest-asyncio
else:
    print("Local environment detected. Ensure: pip install -e '.[dev,benchmark]'")

import nest_asyncio
nest_asyncio.apply()

print("Setup complete")


## Authentication


In [ ]:
import os

# HuggingFace token for gated model access
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except ImportError:
    pass

assert os.environ.get("HF_TOKEN"), "HF_TOKEN not set! Add it to Colab Secrets or environment."
print("Authentication complete")


## Load Model & Build Graph


In [ ]:
import time
from cap_agent.models.medgemma import get_model_and_processor
from cap_agent.agent.graph import build_cap_agent_graph
from cap_agent.agent.state import build_initial_state

# Load MedGemma (lazy singleton - only loads once, includes warm-up)
model, processor = get_model_and_processor()
print(f"Model loaded on {model.device}")

# Build the 8-node LangGraph pipeline
graph = build_cap_agent_graph()
print("Graph compiled: 8 nodes, conditional routing at contradiction check")


## What Are Thinking Tokens?

MedGemma uses special delimiter tokens to structure its internal reasoning:

- **`<unused94>`** marks the **start** of thinking
- **`<unused95>`** marks the **end** of thinking
- Content between these tokens is the model's **internal reasoning** — not shown to users by default

The function `extract_thinking()` in `cap_agent.models.medgemma` separates thinking from the
visible conclusion. This separation is critical for:

1. **Transparency:** Clinicians can inspect *why* the model reached a conclusion
2. **Auditability:** Thinking traces are stored in `state["thinking_traces"]` for review
3. **Confidence calibration:** Better reasoning correlates with higher confidence

### When Thinking Is Enabled vs Disabled

| Call Type | Thinking | Rationale |
|-----------|----------|-----------|
| CXR classification/localization/longitudinal | Enabled | Complex visual reasoning benefits from chain-of-thought |
| Contradiction resolution (Strategies A-D) | Enabled | Cross-modal reasoning requires explicit deliberation |
| EHR/Lab extraction (5 calls) | Disabled | JSON-only output, faster inference |
| Clinician summary | Disabled | Free-text generation, no reasoning needed |
| Strategy E resolution | N/A | Pure Python logic, no MedGemma call |


## Load Contradiction Cases

Group B contains cross-modal contradiction cases (CR-1 through CR-6) that exercise Strategies A-D.
We also load a single Strategy E case (CR-8) for contrast.


In [ ]:
from benchmark_data.cases.registry import get_track2_cases

all_track2 = get_track2_cases()

# Group B: cross-modal contradictions (CR-1 through CR-6)
group_b = [c for c in all_track2 if any(c["case_id"].startswith(prefix) for prefix in ["CR1-", "CR2-", "CR3-", "CR4", "CR5", "CR6"])]
print(f"Loaded {len(group_b)} contradiction cases for reasoning trace analysis")
for c in group_b:
    print(f"  {c['case_id']}: expected contradictions: {c['ground_truth'].get('contradictions', [])}")

# Group C: stewardship (Strategy E) for contrast
group_c_sample = [c for c in all_track2 if c["case_id"].startswith("CR8-")][:1]
print(f"\nLoaded {len(group_c_sample)} Strategy E case for comparison")


In [ ]:
# Clinical output renderer — formats structured_output as a readable document
def render_clinical_output(result, scenario_title="Pipeline Output"):
    """Render structured_output as a formatted clinical document using ANSI codes."""
    ESC = chr(27)
    B = f"{ESC}[1m"       # Bold
    R = f"{ESC}[91m"      # Red
    Y = f"{ESC}[93m"      # Yellow
    G = f"{ESC}[92m"      # Green
    C = f"{ESC}[96m"      # Cyan
    X = f"{ESC}[0m"       # Reset

    so = result.get("structured_output", {})
    sections = so.get("sections", {})

    print(f"\n{B}{C}{'=' * 70}")
    print(f"  CLINICAL DECISION SUPPORT — {scenario_title}")
    print(f"{'=' * 70}{X}")
    print(f"{Y}AI-generated observations for clinician review — not a substitute for clinical judgement.{X}\n")

    # 1. PATIENT
    s1 = sections.get("1_patient", {})
    demo = s1.get("demographics", {})
    print(f"{B}{C}1. PATIENT{X}")
    print(f"  {demo.get('age', '?')}yo {demo.get('sex', '?')}")
    allergy_list = demo.get("allergies", [])
    if allergy_list:
        for a in allergy_list:
            if isinstance(a, dict):
                print(f"  {R}Allergy: {a.get('drug', '?')} — {a.get('reaction_type', '?')} ({a.get('severity', '?')}){X}")
            else:
                print(f"  {R}Allergy: {a}{X}")
    else:
        print(f"  {G}No known drug allergies{X}")
    combos = demo.get("comorbidities", [])
    if combos:
        print(f"  Comorbidities: {', '.join(combos)}")
    if demo.get("pregnancy"):
        print(f"  {R}PREGNANT{X}")
    print(f"  Oral tolerance: {'Yes' if demo.get('oral_tolerance', True) else R + 'No' + X}")
    travel = demo.get("travel_history", [])
    if travel:
        print(f"  {Y}Travel: {', '.join(travel)}{X}")
    print()

    # 2. SEVERITY
    s2 = sections.get("2_severity", {})
    curb = s2.get("curb65", {})
    sev = curb.get("severity_tier", "?")
    sev_color = R if sev == "high" else (Y if sev == "moderate" else G)
    score = curb.get("curb65")
    score_str = f"CURB65={score}" if score is not None else f"CRB65={curb.get('crb65', '?')} (urea unavailable)"
    print(f"{B}{C}2. SEVERITY{X}")
    print(f"  {score_str}  {sev_color}{B}{sev.upper()}{X}")
    print(f"  C={curb.get('c','?')} U={curb.get('u','?')} R={curb.get('r','?')} B={curb.get('b','?')} 65={curb.get('age_65','?')}")
    missing = curb.get("missing_variables", [])
    if missing:
        print(f"  {Y}Missing: {', '.join(missing)}{X}")
    poc = s2.get("place_of_care", {})
    if poc:
        print(f"  Place of care: {poc.get('recommendation', '?')}")
    print()

    # 3. CXR
    s3 = sections.get("3_cxr", {})
    cxr = s3.get("findings", {})
    print(f"{B}{C}3. CHEST X-RAY{X}")
    for finding in ["consolidation", "pleural_effusion", "cardiomegaly", "edema", "atelectasis"]:
        f = cxr.get(finding, {})
        if not isinstance(f, dict):
            continue
        present = f.get("present", False)
        conf = f.get("confidence", "?")
        if present:
            loc = f.get("location", "")
            bbox = f.get("bounding_box")
            line = f"  {R}PRESENT{X} {finding} ({conf} confidence)"
            if loc:
                line += f" at {loc}"
            if bbox:
                line += f" [bbox: {bbox}]"
            print(line)
        else:
            print(f"  {G}absent{X}  {finding} ({conf} confidence)")
    iq = cxr.get("image_quality", {})
    if iq:
        print(f"  Quality: {iq.get('projection','?')} / {iq.get('rotation','?')} / {iq.get('penetration','?')}")
    longit = cxr.get("longitudinal_comparison")
    if longit:
        print(f"  {B}Longitudinal:{X}")
        for fn, ch in longit.items():
            if isinstance(ch, dict):
                print(f"    {fn}: {ch.get('change','?')} — {ch.get('description','')}")
    print()

    # 4. KEY BLOODS
    s4 = sections.get("4_key_bloods", {})
    labs = s4.get("values", {})
    print(f"{B}{C}4. KEY BLOODS{X}")
    for test, data in (labs or {}).items():
        if not isinstance(data, dict):
            continue
        val = data.get("value", "?")
        unit = data.get("unit", "")
        ref = data.get("reference_range", "")
        abn = data.get("abnormal_flag", False)
        color = R if abn else G
        flag = " ABNORMAL" if abn else ""
        print(f"  {color}{test}: {val} {unit}{flag}{X}  (ref: {ref})")
    print()

    # 5. CONTRADICTIONS
    s5 = sections.get("5_contradiction_alert", {})
    alerts = s5.get("alerts", [])
    informational = s5.get("informational", [])
    resolutions = s5.get("resolutions", [])
    n_total = s5.get("detected", 0)
    print(f"{B}{C}5. CONTRADICTION ALERTS{X}")
    if n_total == 0:
        print(f"  {G}No contradictions detected — findings concordant{X}")
    else:
        print(f"  {n_total} contradiction(s) detected")
        for c in alerts:
            conf = c.get("confidence", "?")
            badge_color = R if conf == "high" else Y
            print(f"  {badge_color}[{conf.upper()}]{X} {c.get('rule_id','?')}: {c.get('pattern','')}")
            if c.get("evidence_for"):
                print(f"    For: {c['evidence_for']}")
            if c.get("evidence_against"):
                print(f"    Against: {c['evidence_against']}")
            rec = c.get("recommendation")
            if rec:
                print(f"    {B}Recommendation:{X} {rec}")
        for c in informational:
            print(f"  {G}[LOW]{X} {c.get('rule_id','?')}: {c.get('pattern','')}")
        if resolutions:
            print(f"  {B}Resolutions:{X}")
            for r in resolutions:
                print(f"    {r[:200]}")
    print()

    # 6. TREATMENT
    s6 = sections.get("6_treatment_pathway", {})
    abx = s6.get("antibiotic", {})
    print(f"{B}{C}6. TREATMENT{X}")
    print(f"  First-line: {abx.get('first_line', 'N/A')}")
    print(f"  Dose/route: {abx.get('dose_route', 'N/A')}")
    if abx.get("allergy_adjustment"):
        print(f"  {Y}Allergy adj: {abx['allergy_adjustment']}{X}")
    if abx.get("atypical_cover"):
        print(f"  Atypical: {abx['atypical_cover']}")
    if abx.get("renal_adjustment"):
        print(f"  {Y}Renal: {abx['renal_adjustment']}{X}")
    cortico = s6.get("corticosteroid")
    if cortico:
        print(f"  Corticosteroid: {cortico}")
    steward = abx.get("stewardship_notes", [])
    for note in steward:
        print(f"  {Y}Stewardship: {note}{X}")
    inv = s6.get("investigations", {})
    if inv:
        print(f"  {B}Investigations:{X}")
        for name, detail in inv.items():
            if isinstance(detail, dict):
                rec = "Recommended" if detail.get("recommended") else "Not indicated"
                print(f"    {name}: {rec} — {detail.get('reasoning', '')[:100]}")
    print()

    # 7. DATA GAPS
    s7 = sections.get("7_data_gaps", {})
    gaps = s7.get("gaps", [])
    print(f"{B}{C}7. DATA GAPS{X}")
    if gaps:
        for g in gaps:
            print(f"  {Y}- {g}{X}")
    else:
        print(f"  {G}None identified{X}")
    print()

    # 8. MONITORING
    s8 = sections.get("8_monitoring", {})
    plan = s8.get("plan", {})
    print(f"{B}{C}8. MONITORING{X}")
    crp_trend = plan.get("crp_trend")
    if crp_trend:
        adm = crp_trend.get("admission_value", "?")
        cur = crp_trend.get("current_value", "?")
        pct = crp_trend.get("percent_change", "?")
        trend = crp_trend.get("trend", "?")
        sr = crp_trend.get("flag_senior_review", False)
        sr_color = R if sr else G
        print(f"  CRP: {adm} -> {cur} mg/L ({pct}% change, {trend})")
        print(f"  Senior review: {sr_color}{sr}{X}")
    tr = plan.get("treatment_response")
    if tr:
        reassess = tr.get("reassess_needed", False)
        ra_color = R if reassess else G
        print(f"  Treatment response: {ra_color}reassess_needed={reassess}{X}")
        for action in tr.get("actions", []):
            print(f"    - {action}")
    dc = plan.get("discharge_criteria_met")
    if dc is not None:
        dc_color = G if dc else R
        dc_str = "MET" if dc else "NOT MET"
        print(f"  Discharge: {dc_color}{dc_str}{X}")
    dc_detail = plan.get("discharge_criteria_details", {})
    if dc_detail:
        for check, passed_val in dc_detail.items():
            if check.startswith("_"):
                continue
            chk_color = G if passed_val else R
            chk_str = "PASS" if passed_val else "FAIL"
            print(f"    {chk_color}[{chk_str}]{X} {check}")
    print(f"  Next review: {plan.get('next_review', 'N/A')}")
    print()

    # PROVENANCE
    prov = so.get("provenance", {})
    if prov:
        print(f"{B}{C}PROVENANCE{X}")
        tools = prov.get("extraction_tools", {})
        sources = prov.get("data_sources", {})
        for tool_name, pipeline in tools.items():
            src = sources.get(tool_name, [])
            print(f"  {tool_name}: {pipeline} ({', '.join(src) if src else 'N/A'})")

    print(f"\n{C}{'=' * 70}{X}\n")

print("render_clinical_output() defined")


## Run All Cases

Run each Group B case (cross-modal contradictions, Strategies A-D) plus one Strategy E case
through the full pipeline. The focus is on capturing thinking traces and resolution results.


In [ ]:
all_results = []

for i, case in enumerate(group_b + group_c_sample):
    case_id = case["case_id"]
    print(f"[{i+1}/{len(group_b) + len(group_c_sample)}] Running {case_id}...")

    initial_state = build_initial_state(case)
    start = time.time()
    result = None
    for chunk in graph.stream(initial_state, stream_mode="values"):
        result = chunk
    elapsed = time.time() - start

    all_results.append({
        "case_id": case_id,
        "result": result,
        "elapsed": elapsed,
        "ground_truth": case["ground_truth"],
    })
    n_traces = len(result.get("thinking_traces", []))
    n_contradictions = len(result.get("contradictions_detected", []))
    print(f"  Done in {elapsed:.1f}s — {n_contradictions} contradictions, {n_traces} thinking traces")

print(f"\nAll {len(all_results)} cases complete")


In [ ]:
def display_thinking_trace(result, case_id):
    """Display thinking traces and resolution results with formatting."""
    ESC = chr(27)
    B = f"{ESC}[1m"
    C = f"{ESC}[96m"
    Y = f"{ESC}[93m"
    G = f"{ESC}[92m"
    M = f"{ESC}[95m"
    R = f"{ESC}[91m"
    X = f"{ESC}[0m"

    traces = result.get("thinking_traces", [])
    resolutions = result.get("resolution_results", [])
    contradictions = result.get("contradictions_detected", [])

    print(f"\n{B}{C}{'=' * 70}")
    print(f"  REASONING TRACE: {case_id}")
    print(f"{'=' * 70}{X}\n")

    # Show contradictions detected
    print(f"{B}Contradictions Detected:{X}")
    for c in contradictions:
        conf = c.get("confidence", "?")
        conf_color = R if conf == "high" else (Y if conf == "moderate" else G)
        print(f"  {c.get('rule_id', '?')}: {c.get('description', '?')}")
        print(f"  Confidence: {conf_color}{conf}{X}")
        print(f"  Strategy: {c.get('resolution_strategy', '?')}")
        print()

    # Show thinking traces
    if traces:
        print(f"{B}{M}Internal Thinking (MedGemma):{X}")
        print(f"{M}{'─' * 60}{X}")
        for i, trace in enumerate(traces):
            # Truncate very long traces for readability
            display = trace[:2000] + "..." if len(trace) > 2000 else trace
            print(f"{M}{display}{X}")
            if i < len(traces) - 1:
                print(f"{M}{'─' * 40}{X}")
        print(f"{M}{'─' * 60}{X}\n")
    else:
        print(f"{Y}No thinking traces captured (Strategy E uses deterministic logic){X}\n")

    # Show resolution results
    if resolutions:
        print(f"{B}{G}Resolution Conclusions:{X}")
        for res in resolutions:
            display = res[:1500] + "..." if len(res) > 1500 else res
            print(f"  {G}{display}{X}")
    else:
        print(f"{Y}No GPU-based resolution (Strategy E){X}")

print("display_thinking_trace() defined")


## Strategy A: Zone-Specific CXR Re-analysis

**Used for:** CR-1 (CXR negative + focal respiratory signs) and CR-5 (bilateral CXR + unilateral exam)

When the chest X-ray and clinical examination disagree on the location or presence of pathology,
MedGemma re-analyzes the CXR focusing on specific anatomical zones mentioned in the clinical
findings.

**What the thinking trace reveals:**
- Which lung zones are being examined
- What opacity patterns are being considered (subtle ground-glass, retrocardiac consolidation)
- How the model calibrates its confidence based on image quality and finding subtlety
- Whether the re-analysis changes the initial CXR interpretation


In [ ]:
cr1_results = [r for r in all_results if r["case_id"].startswith("CR1-")]
for r in cr1_results:
    display_thinking_trace(r["result"], r["case_id"])


### CR-5: Bilateral CXR + Unilateral Exam

A variant of Strategy A where the CXR shows bilateral consolidation but clinical examination
finds signs only on one side. The thinking trace shows how MedGemma reconciles this.


In [ ]:
cr5_results = [r for r in all_results if r["case_id"] == "CR5"]
for r in cr5_results:
    display_thinking_trace(r["result"], r["case_id"])


## Strategy B: Temporal Context Assessment

**Used for:** CR-2 (CXR negative + high CRP)

When the CXR shows no consolidation but inflammatory markers (CRP) are significantly elevated,
MedGemma evaluates whether the clinical timeline explains the discrepancy.

**What the thinking trace reveals:**
- Consideration of early vs late presentation (CXR sensitivity increases over time)
- Whether the CRP level is consistent with the reported symptom duration
- Alternative inflammatory causes that could explain the CRP elevation
- A temporal hypothesis for why imaging may lag behind lab findings


In [ ]:
cr2_results = [r for r in all_results if r["case_id"].startswith("CR2-")]
for r in cr2_results:
    display_thinking_trace(r["result"], r["case_id"])


## Strategy C: Differential Diagnosis Reasoning

**Used for:** CR-3 (CXR consolidation + low CRP) and CR-6 (pleural effusion without consolidation)

When imaging and laboratory findings point in different directions, MedGemma generates a
differential diagnosis list, reasoning through each alternative that could explain the
discrepancy.

**What the thinking trace reveals:**
- A structured differential diagnosis list
- Reasoning about each alternative (e.g. atelectasis vs true consolidation, transudative vs exudative effusion)
- Consideration of pre-test probability and clinical context
- A final assessment with calibrated confidence


In [ ]:
cr3_results = [r for r in all_results if r["case_id"] == "CR3-HIGH"]
cr6_results = [r for r in all_results if r["case_id"] == "CR6"]

for r in cr3_results + cr6_results:
    display_thinking_trace(r["result"], r["case_id"])


## Strategy D: Severity Override Reasoning

**Used for:** CR-4 (low severity + override triggers like immunosuppression or bilateral findings)

CURB65 is a validated severity score, but it has known blind spots. Immunosuppressed patients
and those with bilateral involvement may have low CURB65 scores despite high clinical risk.
MedGemma weighs these additional risk factors that CURB65 does not capture.

**What the thinking trace reveals:**
- Risk assessment beyond the CURB65 components
- Consideration of immunocompromised state and its implications
- Bilateral involvement significance for disease burden
- Whether the severity tier should be overridden despite the numerical score


In [ ]:
cr4_results = [r for r in all_results if r["case_id"] == "CR4-HIGH"]
for r in cr4_results:
    display_thinking_trace(r["result"], r["case_id"])


## Strategy E: Deterministic Logic (No Thinking Tokens)

**Used for:** Stewardship rules CR-7 through CR-11

Strategy E is fundamentally different from Strategies A-D. Resolution uses **pure Python logic** —
no MedGemma, no GPU, no thinking tokens. The decision is instant and fully deterministic.

This is a key design decision: **not everything needs AI reasoning.** When the clinical rule is
unambiguous and the input data is structured (e.g. "is this organism susceptible to this
antibiotic?"), deterministic logic is:
- Faster (no GPU call)
- More predictable (same input always produces same output)
- Easier to audit (no black-box reasoning to inspect)

The pipeline intelligently routes to Strategy E when AI reasoning would not add value.


In [ ]:
cr8_results = [r for r in all_results if r["case_id"].startswith("CR8-")]
for r in cr8_results:
    display_thinking_trace(r["result"], r["case_id"])
    print("\n  → Strategy E: deterministic resolution, no thinking tokens needed")
    print("  → The system correctly distinguishes when AI reasoning adds value vs when rules suffice")


## Thinking → Confidence Mapping

The pipeline maps thinking traces to confidence levels through a structured process:

1. **Detection assigns initial confidence** based on rule-specific logic (e.g. CR-1 with both
   crackles AND bronchial breathing → "high", crackles only → "moderate")
2. **Resolution may update confidence** via `_parse_resolution_confidence()` in nodes.py,
   which uses regex to extract `CONFIDENCE: high/moderate/low` from MedGemma output
3. **Alert tiering** uses the final confidence level:
   - **High/Moderate confidence** → action alerts (require clinician attention)
   - **Low confidence** → informational alerts (for awareness only)

This mapping ensures that stronger reasoning → higher confidence → more prominent alerts.
The system never suppresses alerts — it only adjusts their prominence.


In [ ]:
print("=== CONFIDENCE DISTRIBUTION ===\n")
print(f"{'Case ID':<15} {'Rule':<8} {'Strategy':<12} {'Confidence':<12} {'Thinking Tokens'}")
print("-" * 70)

for r in all_results:
    case_id = r["case_id"]
    contradictions = r["result"].get("contradictions_detected", [])
    traces = r["result"].get("thinking_traces", [])
    trace_len = sum(len(t) for t in traces)
    for c in contradictions:
        rule = c.get("rule_id", "?")
        strategy = c.get("resolution_strategy", "?")
        confidence = c.get("confidence", "?")
        print(f"{case_id:<15} {rule:<8} {strategy:<12} {confidence:<12} {trace_len} chars")


In [ ]:
import plotly.graph_objects as go

strategies = []
confidences = []
case_ids = []

for r in all_results:
    for c in r["result"].get("contradictions_detected", []):
        strategies.append(c.get("resolution_strategy", "?"))
        confidences.append(c.get("confidence", "?"))
        case_ids.append(r["case_id"])

# Count by strategy and confidence
from collections import Counter
strategy_conf = Counter(zip(strategies, confidences))

unique_strategies = sorted(set(strategies))
conf_levels = ["high", "moderate", "low"]
colors = {"high": "#dc2626", "moderate": "#f59e0b", "low": "#22c55e"}

fig = go.Figure()
for conf in conf_levels:
    counts = [strategy_conf.get((s, conf), 0) for s in unique_strategies]
    if any(c > 0 for c in counts):
        fig.add_trace(go.Bar(
            x=unique_strategies, y=counts,
            name=conf.capitalize(),
            marker_color=colors[conf],
        ))

fig.update_layout(
    title="Confidence Distribution by Resolution Strategy",
    xaxis_title="Resolution Strategy",
    yaxis_title="Count",
    barmode="group",
    template="plotly_white",
    height=400,
)
fig.show()


## Summary: Model Transparency in Clinical AI

This notebook demonstrates several important principles:

1. **MedGemma doesn't just produce answers — it reasons.** The thinking tokens reveal a
   structured deliberation process that considers multiple hypotheses before reaching a conclusion.

2. **Different clinical scenarios require different reasoning strategies.** Zone-specific CXR
   re-analysis (Strategy A) is fundamentally different from differential diagnosis generation
   (Strategy C) or severity override assessment (Strategy D).

3. **The system knows when to use AI reasoning vs deterministic rules.** Strategy E (CR-7 through
   CR-11) correctly avoids GPU calls when the clinical rule is unambiguous. This is not a
   limitation — it is a design decision that improves reliability and speed.

4. **Thinking tokens provide a window into model behaviour that supports clinical trust.**
   A clinician reviewing a contradiction alert can inspect the reasoning that led to it,
   rather than accepting a black-box recommendation.

5. **Confidence calibration from reasoning quality enables appropriate alert tiering.**
   High-quality reasoning produces high-confidence alerts that demand attention; uncertain
   reasoning produces informational alerts for awareness only.


## Verification Checklist

Verify that thinking traces were captured where expected and absent where not expected.


In [ ]:
print("=== VERIFICATION CHECKLIST ===\n")

checks = [
    ("All Group B cases ran successfully", all(r["result"] is not None for r in all_results if r["case_id"] != "CR8-FIRE")),
    ("CR-1 cases have thinking traces", all(
        len(r["result"].get("thinking_traces", [])) > 0
        for r in all_results if r["case_id"].startswith("CR1-")
    )),
    ("CR-2 cases have thinking traces", all(
        len(r["result"].get("thinking_traces", [])) > 0
        for r in all_results if r["case_id"].startswith("CR2-")
    )),
    ("CR-4 has thinking traces", any(
        len(r["result"].get("thinking_traces", [])) > 0
        for r in all_results if r["case_id"] == "CR4-HIGH"
    )),
    ("Strategy E case has NO thinking traces", all(
        len(r["result"].get("thinking_traces", [])) == 0
        for r in all_results if r["case_id"].startswith("CR8-")
    )),
    ("All Strategies A-D cases detected contradictions", all(
        len(r["result"].get("contradictions_detected", [])) > 0
        for r in all_results if not r["case_id"].startswith("CR8-")
    )),
    ("Confidence levels assigned to all contradictions", all(
        all(c.get("confidence") in ("high", "moderate", "low") for c in r["result"].get("contradictions_detected", []))
        for r in all_results
    )),
]

passed = 0
for desc, ok in checks:
    status = "PASS" if ok else "FAIL"
    icon = "+" if ok else "x"
    print(f"  [{icon}] {status}: {desc}")
    if ok:
        passed += 1

print(f"\n{passed}/{len(checks)} checks passed")
